In [1]:
import sys, platform, importlib.metadata, json
print("AUDIT_KERNEL="+json.dumps({"python":sys.version,"executable":sys.executable,"platform":platform.platform(),"packages":dict(sorted((d.metadata["Name"],d.version) for d in importlib.metadata.distributions()))}))

AUDIT_KERNEL={"python": "3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]", "executable": "C:\\Users\\Miakhlov\\Documents\\GitHub\\Computational chemistry_course\\tmp\\course-restructure-20261118\\scripts\\environment\\.venv-chem\\Scripts\\python.exe", "platform": "Windows-10-10.0.19045-SP0", "packages": {"Jinja2": "3.1.6", "MarkupSafe": "3.0.4", "PyYAML": "6.0.3", "Pygments": "2.21.0", "Send2Trash": "2.2.0", "altair": "6.3.0", "anyio": "4.16.0", "argon2-cffi": "25.1.0", "argon2-cffi-bindings": "26.2.0", "arrow": "1.4.0", "asttokens": "3.0.2", "async-lru": "2.4.0", "attrs": "26.1.0", "babel": "2.18.0", "beautifulsoup4": "4.15.0", "bleach": "6.4.0", "certifi": "2026.10.23", "cffi": "2.1.1", "charset-normalizer": "3.5.2", "click": "8.5.1", "cloudpickle": "3.1.3", "colorama": "0.4.6", "comm": "0.2.3", "contourpy": "1.4.0", "cycler": "0.12.1", "debugpy": "1.8.22", "defusedxml": "0.7.1", "executing": "2.3.0", "fastjsonschema": "2.22.3", "fonttools": "4.68.0

# Разбор A. Электронная модель

Механическое поле не описывает перераспределение электронов. Прежде чем читать квантовый результат истрадефиллина, воспроизведём ограниченный RHF-контроль H₂. Гипотеза: энергия зависит от межъядерного расстояния, а код должен одновременно контролировать плотность и энергию. H₂ проверяет метод, но не сообщает свойства A2A-лиганда или аторвастатина.

Запустите существующее прозрачное ядро h2_scf на 0.74, 1.0 и 1.5 Å. Сохраните трассы итераций, проверьте известный контроль около −1.11676 Eh на 0.74 Å. Затем попытайтесь ограничить число итераций единицей и зафиксируйте отказ. Обсудите невозможность переноса энергий H₂ на лекарственные объекты.

In [2]:
from pathlib import Path
import sys,json,math
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'programm/course.json').is_file())
sys.path.insert(0,str(ROOT/'practicuum/shared'))
from course_support import DATA,compound,esol,workspace,write_json,init_dossier
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
def publish_plot(fig,name):
    fig.savefig(OUT/name,dpi=120,bbox_inches='tight')
    display(fig)
    plt.close(fig)
OUT=workspace('M07-L01','case')


Сначала проверяем входы и единицы. Словари сохраняют названия полей вместе со значениями; список сохраняет соответствие записей. Выведите промежуточный результат перед дальнейшим анализом.

In [3]:
from h2_scf import rhf_h2
rows=[rhf_h2(r) for r in [.74,1.,1.5]]
assert all(r['converged'] for r in rows)
assert abs(rows[0]['energy_hartree']-(-1.1167593))<1e-5
display([{k:r[k] for k in ['distance_angstrom','energy_hartree','cycles']} for r in rows])


[{'distance_angstrom': 0.74,
  'energy_hartree': -1.1167593073781579,
  'cycles': 2},
 {'distance_angstrom': 1.0,
  'energy_hartree': -1.0661086491853147,
  'cycles': 2},
 {'distance_angstrom': 1.5,
  'energy_hartree': -0.9108735543822694,
  'cycles': 2}]

Теперь сравним модель с контрольным случаем. `assert` останавливает расчёт при нарушении контракта; это защита от ошибочного продолжения, а не оценка научного понимания.

In [4]:
try:rhf_h2(.74,max_cycles=1)
except RuntimeError as error:print('Expected nonconvergence:',error)
else:raise AssertionError('One cycle accepted')
fig,ax=plt.subplots();ax.plot([r['distance_angstrom'] for r in rows],[r['energy_hartree'] for r in rows],'o-')
ax.set(xlabel='H-H / Angstrom',ylabel='RHF/STO-3G energy / Eh');publish_plot(fig,'h2.png')


Expected nonconvergence: SCF did not converge


<Figure size 640x480 with 1 Axes>

Сохраним именно полученный результат. Не переносите этот вывод на другую химическую форму, мишень или экспериментальные условия без отдельной проверки.

In [5]:
result={'control':'H2 only','runs':rows,'kind':'calculated','drug_QM':'not inferred from H2'}


In [6]:
write_json(OUT/'result.json',result)
from dossier_records import append_record
receipt=append_record(init_dossier(),'M07-L01','A',result,ROOT,inputs=[])
print('Versioned evidence:',receipt)


Versioned evidence: C:\Users\Miakhlov\Documents\GitHub\Computational chemistry_course\tmp\course-restructure-20261118\practicuum\evidence\20261118-full-course-windows\workspace\my_research\evidence\M07-L01\20261118T141603285049Z-A-5fb2d9877451469d965fabfe61d86ff3.json


Кривая показывает зависимость энергии в выбранном базисе, не свободную энергию реакции. Для реального B нужно отдельно задать заряд, spin=2S, геометрию и метод. Следующий урок проверит именно паспорт полного расчёта.

Перед переходом ответьте: какие результаты получены вычислением, какие заданы условно, и какая независимая проверка ещё отсутствует?